## Data Cleaning am Beispiel der Datenjagd

In [ ]:
# pandas library für data manipulation & analysis importieren
import pandas as pd 

In [ ]:
datenjagd =pd.read_csv("Data/Datenjagd.csv")
datenjagd

Bevor Daten analysiert werden, sollten Variablen nach ihrer Bedeutung und Messart gruppiert werden. Nicht jede Zahl beschreibt dieselbe Art von Information. Deshalb braucht es ein Verständnis der Variablentypen.

*Namenskonventionen im Datensatz*

* _time_s → **Zeitdauer** (Messung in Sekunden)
  Beispiele: Reaktionszeit, Bewegungsdauer, Aktivitätsdauer

* _time → **Zeitpunkt** (Uhrzeit)
  Beispiele: Einschlafzeit, Aufstehzeit

* _level → **Subjektive Skala** (Selbsteinschätzung)
  Beispiele: Stresslevel, Hungerlevel, Konzentration

* _count → **Diskrete Zählwerte**
  Beispiele: Anzahl Schritte, Tabs, Apps

* \_estimated_ → **Geschätzte Werte**
  Beispiele: geschätzte Distanz oder Temperatur




In [ ]:
datenjagd.columns = [
    "id",
    "shoe_type",
    "pen_count",
    "seat_distance_board_m",
    "items_on_desk_count",
    "hunger_level",
    "temperature_feeling_level",
    "key_count",
    "favorite_break_drink",
    "wall_distance_estimated_m",
    "favorite_season",
    "seat_satisfaction_level",
    "concentration_level",
    "door_walk_time_s",
    "corridor_round_steps",
    "fatigue_level",
    "current_mood",
    "count_1_to_20_time_s",
    "middle_name_letter_count",
    "chair_height_estimated_cm",
    "favorite_subject",
    "did_eat_breakfast",
    "transport_wheel_count",
    "browser_tab_count",
    "first_name_write_time_s",
    "desk_height_mm",
    "sleep_time",
    "jumping_jacks_10_time_s",
    "has_phone_case",
    "wake_time_today",
    "room_temperature_estimated_c",
    "stress_level",
    "phone_width_mm",
    "wake_time_weekend",
    "app_count_phone",
    "pen_length_cm"
]
datenjagd.drop(columns=["id"], inplace=True)

In [ ]:
datenpunkte_vorher = datenjagd.shape[0]
datenjagd.drop_duplicates(inplace=True)
datenpunkte_nachher = datenjagd.shape[0]    
print(f"Anzahl der Duplikate: {datenpunkte_vorher-datenpunkte_nachher}")

In [ ]:
datenjagd.info()

In [ ]:
datenjagd

Die folgende Aufteilung dient dazu, die Spalten später unterschiedlich zu verarbeiten:

- kategoriale Variablen kodieren,
- binäre Variablen speziell behandeln,
- ordinale Werte in einer Reihenfolge berücksichtigen,
- Zeitvariablen als Zeitdaten zu interpretieren,
- numerische Werte normalisieren oder analysieren.

Damit wird die Datenvorbereitung strukturierter und übersichtlicher.

In [ ]:
categorical_cols = [
    "shoe_type",
    "favorite_break_drink",
    "favorite_season",
    "favorite_subject",
    "current_mood",
    "temperature_feeling_level"
]

binary_cols = [
    "did_eat_breakfast",
    "has_phone_case"
]

ordinal_cols = [
    "hunger_level",
    "seat_satisfaction_level",
    "concentration_level",
    "fatigue_level",
    "stress_level"
]

time_cols = [
    "sleep_time",
    "wake_time_today",
    "wake_time_weekend"
]

numeric_cols = [
    "pen_count",
    "seat_distance_board_m",
    "items_on_desk_count",
    "key_count",
    "wall_distance_estimated_m",
    "door_walk_time_s",
    "corridor_round_steps",
    "count_1_to_20_time_s",
    "middle_name_letter_count",
    "chair_height_estimated_cm",
    "transport_wheel_count",
    "browser_tab_count",
    "first_name_write_time_s",
    "desk_height_mm",
    "jumping_jacks_10_time_s",
    "room_temperature_estimated_c",
    "phone_width_mm",
    "app_count_phone",
    "pen_length_cm"
]

mapping = {
    "ja": True,
    "nein": False,
    "yes": True,
    "no": False,
    "true": True,
    "false": False
}

In [ ]:
# Konvertieren der Datentypen
datenjagd[categorical_cols] = datenjagd[categorical_cols].astype("category") # Konvertieren der kategorischen Spalten
for col in binary_cols: # Konvertieren der binären Spalten
    datenjagd[col] = (
    datenjagd[col]
      .astype(str)
      .str.strip()
      .str.lower()
      .map(mapping)
)
datenjagd[ordinal_cols] = datenjagd[ordinal_cols].astype("float64") # Konvertieren der ordinalen Spalten in numerische Werte
datenjagd[numeric_cols] = datenjagd[numeric_cols].apply(pd.to_numeric, errors="coerce")

In [ ]:
for col in categorical_cols:
    print(f"{datenjagd[col].value_counts(dropna=False)}\n")

In [ ]:
datenjagd

### Zeitspalten — verstehen, bevor Transformationen durchgeführt werden

Bevor Zeitvariablen transformiert oder analysiert werden, muss zuerst geklärt werden, **welche Art von Zeitinformation** vorliegt.
Nicht jede Angabe, die wie eine Uhrzeit aussieht, bedeutet statistisch dasselbe.

**Es gibt zwei grundlegende Konzepte:**

**1. Zeitpunkte (Uhrzeiten)**
Beschreiben, *wann* etwas passiert ist.

Beispiele:

* sleep_time
* wake_time_today
* wake_time_weekend

Eigenschaften:

* Position auf einer täglichen Zeitachse
* Differenzen zwischen Zeitpunkten ergeben eine Dauer
* Mittelwerte von Uhrzeiten sind oft nicht sinnvoll

Typischer Fehler:
Der Durchschnitt von 23:30 und 01:30 ist **nicht** 12:30.


**2. Zeitdauern**

Beschreiben, *wie lange* etwas dauert.

Beispiele:

* door_walk_time_s
* jumping_jacks_10_time_s
* count_1_to_20_time_s

Eigenschaften:

* echte messbare Grössen
* Mittelwerte und Vergleiche sind sinnvoll
* direkt statistisch auswertbar

--- 

Diese Unterscheidung ist wichtig, weil falsche Ergebnisse entstehen können, wenn Uhrzeiten wie normale Zahlen behandelt werden:

* Zeitpunkte müssen zuerst sinnvoll umgerechnet werden
* Zeitdauern können direkt analysiert werden
* Visualisierungen verhalten sich unterschiedlich je nach Interpretation

Gute Datenanalyse beginnt mit dem Verständnis der Bedeutung einer Variable — nicht mit sofortiger Transformation.


In [ ]:
datenjagd[time_cols]

In [ ]:
datenjagd[time_cols] = datenjagd[time_cols].apply(pd.to_datetime, errors="coerce")

In [ ]:
datenjagd.dtypes
datenjagd.isna().sum()

In [ ]:
datenjagd[time_cols]

### Umgang mit NaN-Werten und erste Datenprüfung

Fehlende Werte (`NaN`) sind ein normaler Bestandteil realer Datensätze.
Bevor Analysen durchgeführt werden, sollte bewusst entschieden werden, wie mit fehlenden Informationen umgegangen wird.

**Mögliche Strategien für NaN-Werte**

* Entfernen von Zeilen mit NaN-Werten: sinnvoll bei wenigen fehlenden Einträgen oder wenn vollständige Daten zwingend nötig sind
* Ersetzen durch statistische Kennwerte: z. B. Mittelwert oder Median bei numerischen Spalten
* Ersetzen durch spezielle Kategorien: z. B. `"Unbekannt"` oder `"Keine Angabe"` bei kategorialen Variablen


In [ ]:
# Zeilen mit zu vielen NaN-Werten entfernen:
max_missing = 3
keep = datenjagd.isna().sum(axis=1) <= max_missing
datenjagd = datenjagd.loc[keep].copy()

# Typgerechte Imputation
datenjagd[categorical_cols] = datenjagd[categorical_cols].fillna("Unbekannt")
datenjagd[binary_cols] = datenjagd[binary_cols].fillna(0)
datenjagd[ordinal_cols] = datenjagd[ordinal_cols].fillna(datenjagd[ordinal_cols].median())
datenjagd[numeric_cols] = datenjagd[numeric_cols].fillna(datenjagd[numeric_cols].median())

In [ ]:
datenjagd

### Weitere empfohlene Prüfschritte vor der Analyse

* Ausreisser untersuchen: ungewöhnlich grosse oder kleine Werte können Messfehler oder interessante Sonderfälle sein
* Korrelationen prüfen: zeigen mögliche Zusammenhänge zwischen numerischen Variablen
* Verteilungen analysieren: helfen zu verstehen, ob Daten symmetrisch, schief oder mehrgipflig verteilt sind

Datenanalyse beginnt nicht mit Modellen oder Diagrammen, sondern mit systematischer Datenprüfung.